# Step 3 — Grid Exploration and Frame Averaging

## What this step does

The giant H5 from Step 2 contains N frames where the PSF was at many different sky positions. The goal of Step 3 is to ask: **"for each sky position, what does the PLcam (fiber spectrograph) see?"**

We do this by:
1. **Defining a spatial grid** over the PSF centroid positions
2. **Assigning each frame** to the nearest grid cell based on its PSF centroid (x, y)
3. **Averaging** all PLcam frames in each cell → one representative PLcam image per sky position

The result is a 4D array: `avg_PLcam[iy, ix]` = the averaged PLcam frame when the PSF was at grid position (ix, iy).

```
                 PSF centroid positions             After binning
     y                                              ┌───┬───┬───┐
     ↑    . . . .  . .                             │ 3 │ 7 │ 2 │  nframes per bin
     │     . .. . .                                ├───┼───┼───┤
     │    . . . . . .                              │ 8 │12 │ 5 │
     └──────────────→ x                            ├───┼───┼───┤
                                                   │ 4 │ 6 │ 9 │
                                                   └───┴───┴───┘
                                                     3×3 grid
```

**Output**: `step3_map.h5` — averaged PSFcam + PLcam frames on a spatial grid.
This is the **coupling map** input to spectral extraction (Step 4) and image reconstruction.

## Setup

In [ ]:
import os, sys

TUTORIAL_DIR = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(TUTORIAL_DIR, '..', '..'))

# ── Input: Step 2 giant H5 ────────────────────────────────────────────────────
ALLDATA_H5_NEW = os.path.join(TUTORIAL_DIR, 'tutorial_output_new', 'step2_alldata.h5')
ALLDATA_H5_PRE = os.path.join(TUTORIAL_DIR, 'tutorial_output', 'alldata.h5')
ALLDATA_H5 = ALLDATA_H5_NEW if os.path.exists(ALLDATA_H5_NEW) else ALLDATA_H5_PRE
print(f'Using giant H5: {ALLDATA_H5}')

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(TUTORIAL_DIR, 'tutorial_output_new')
OUTPUT_H5  = os.path.join(OUTPUT_DIR, 'step3_map.h5')

print(f'Output map H5: {OUTPUT_H5}')

## Step 3a — Explore the PSF centroid grid

Before averaging all frames, use `explore_grid` to preview the grid layout and response maps with just a few parameters. This is **fast** because it reads only the centroid/peak metadata (not the full PLcam data).

### Key parameters to tune

| Parameter | Meaning | Typical value |
|-----------|---------|---------------|
| `map_n` | Grid resolution (map_n × map_n bins) | 5–20 |
| `map_width` | Grid FOV in PSFcam pixels (half-width) | 1.5–4 |
| `xc`, `yc` | Grid centre in PSFcam pixels | median centroid |
| `pix2mas` | Plate scale (milliarcseconds per pixel) | 16.2 for Palila |

In [ ]:
import h5py, numpy as np

# First, look at the centroid data to choose good grid parameters
with h5py.File(ALLDATA_H5, 'r') as f:
    centroids  = f['psfcam/centroids'][:]   # (N, 2) — (x, y) in PSFcam pixels
    peaks      = f['psfcam/peaks'][:]       # (N,)   — PSF peak value
    timestamps = f['metadata/timestamps'][:]  # (N,)  — relative seconds

print(f'N frames         : {len(centroids)}')
print(f'Centroid x range : [{centroids[:,0].min():.2f}, {centroids[:,0].max():.2f}] pixels')
print(f'Centroid y range : [{centroids[:,1].min():.2f}, {centroids[:,1].max():.2f}] pixels')
print(f'Centroid x median: {np.nanmedian(centroids[:,0]):.2f}  ← good starting point for xc')
print(f'Centroid y median: {np.nanmedian(centroids[:,1]):.2f}  ← good starting point for yc')

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Centroid scatter
sc = ax1.scatter(centroids[:, 0], centroids[:, 1], c=np.arange(len(centroids)),
                 cmap='viridis', s=15, alpha=0.7)
plt.colorbar(sc, ax=ax1, label='Frame index')
ax1.set_xlabel('PSF centroid x (pixels)')
ax1.set_ylabel('PSF centroid y (pixels)')
ax1.set_title('PSF centroid positions')
ax1.set_aspect('equal')
ax1.axhline(np.nanmedian(centroids[:,1]), color='red', ls='--', label='median y')
ax1.axvline(np.nanmedian(centroids[:,0]), color='orange', ls='--', label='median x')
ax1.legend()

# Peak histogram — used to set quality cuts
ax2.hist(peaks, bins=30)
ax2.set_xlabel('PSF peak value (counts)')
ax2.set_ylabel('N frames')
ax2.set_title('PSF peak distribution  (Strehl proxy)')

plt.tight_layout()
plt.show()

In [ ]:
import PLred.average as average

# ── Grid parameters ───────────────────────────────────────────────────────────
MAP_N     = 5      # 5×5 spatial grid
MAP_WIDTH = 2.0    # grid spans ±2 PSFcam pixels from centre
XC        = 18.26  # grid centre x (PSFcam pixels) — from median above
YC        = 20.50  # grid centre y (PSFcam pixels)
PIX2MAS   = 16.2   # plate scale: 1 PSFcam pixel = 16.2 mas on sky

# ── Optional quality filters ──────────────────────────────────────────────────
# Uncomment to apply:
# TIME_MIN  = 0.0    # ignore frames before this time (seconds)
# TIME_MAX  = 0.5    # ignore frames after this time
# PEAK_MIN  = 3000   # ignore frames with PSF peak below this value
# PEAK_MAX  = None   # no upper limit

# explore_grid (Mode A — fast: reads only centroid metadata, no PLcam)
result = average.explore_grid(
    giant_h5  = ALLDATA_H5,
    map_n     = MAP_N,
    map_width = MAP_WIDTH,
    xc        = XC,
    yc        = YC,
    pix2mas   = PIX2MAS,
    plot      = True,
)

print('\nnframes per bin:')
print(result['nframes_map'])

The `nframes_map` shows how many PLcam frames fell into each grid bin. Bins with 0 frames are empty (the PSF never pointed there).

### Preview a response map at one PLcam pixel

Now run `explore_grid` again with `plcam_pixels` to see what the coupling map looks like at a specific pixel. This reads a thin slice of the PLcam data — still fast.

In [ ]:
# Pixel (py, px) to preview — try a few values to find a fiber with good signal
# py = fiber axis, px = spectral axis (in the cropped ROI)
PREVIEW_PIXELS = [(200, 10)]   # (py, px) in the cropped PLcam frame

result = average.explore_grid(
    giant_h5      = ALLDATA_H5,
    map_n         = MAP_N,
    map_width     = MAP_WIDTH,
    xc            = XC,
    yc            = YC,
    pix2mas       = PIX2MAS,
    plcam_pixels  = PREVIEW_PIXELS,   # which pixel to extract and bin
    plot          = True,
)

# The response map is stored in result['pixel_maps']
for (py, px), rmap in result['pixel_maps'].items():
    print(f'Response map for pixel (py={py}, px={px}) shape: {rmap.shape}')

## Step 3b — Average frames per bin

Once satisfied with the grid parameters, run `average_to_h5` to compute the full average PLcam and PSFcam image for every non-empty bin. This reads all frames and may take a moment.

### Optional: bootstrap for uncertainty estimates

Set `n_bootstrap > 0` to compute variance maps: for each bin, sample the contributing frames with replacement N times. The variance across bootstrap realizations gives a noise estimate.

In [ ]:
average.average_to_h5(
    giant_h5    = ALLDATA_H5,
    outpath     = OUTPUT_H5,
    map_n       = MAP_N,
    map_width   = MAP_WIDTH,
    xc          = XC,
    yc          = YC,
    pix2mas     = PIX2MAS,
    n_bootstrap = 0,     # set to e.g. 10 for variance estimation
    verbose     = True,
)

print('\nAveraging complete.')

## Inspect the averaged H5

In [ ]:
import h5py, numpy as np

with h5py.File(OUTPUT_H5, 'r') as f:
    avg_plcam  = f['avg_PLcam'][:]          # (map_n, map_n, ny, nx)
    avg_psfcam = f['avg_PSFcam'][:]         # (map_n, map_n, h, w)
    nframes    = f['metadata/nframes'][:]   # (map_n, map_n) — frames per bin
    x_mas      = f['metadata/x_mas'][:]    # x grid centres in mas
    y_mas      = f['metadata/y_mas'][:]    # y grid centres in mas

print(f'avg_PLcam shape  : {avg_plcam.shape}   (map_n, map_n, ny, nx)')
print(f'avg_PSFcam shape : {avg_psfcam.shape}  (map_n, map_n, h, w)')
print(f'nframes per bin  :')
print(nframes)
print(f'\nGrid x centres (mas): {x_mas.round(1)}')
print(f'Grid y centres (mas): {y_mas.round(1)}')

## Visualize coupling maps

Use `pixel_map` to extract the 2D response map for any single PLcam pixel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Response map at one pixel
PY, PX = 200, 10   # pixel in the cropped PLcam frame
rmap = average.pixel_map(OUTPUT_H5, py=PY, px=PX)
print(f'Response map shape: {rmap.shape}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Response map
ax = axes[0]
im = ax.imshow(rmap, origin='lower', cmap='hot',
               extent=[x_mas[0], x_mas[-1], y_mas[0], y_mas[-1]])
plt.colorbar(im, ax=ax, label='Mean PLcam counts')
ax.set_xlabel('Sky x (mas)')
ax.set_ylabel('Sky y (mas)')
ax.set_title(f'Coupling map  pixel (py={PY}, px={PX})')

# Frames per bin
ax = axes[1]
im2 = ax.imshow(nframes, origin='lower', cmap='Blues',
                extent=[x_mas[0], x_mas[-1], y_mas[0], y_mas[-1]])
plt.colorbar(im2, ax=ax, label='N frames')
ax.set_xlabel('Sky x (mas)')
ax.set_ylabel('Sky y (mas)')
ax.set_title('Frames per bin')

# Averaged PSFcam frames for the two best-covered bins
ax = axes[2]
best_bin = np.unravel_index(np.argmax(nframes), nframes.shape)
ax.imshow(avg_psfcam[best_bin[0], best_bin[1]], origin='lower', cmap='inferno')
ax.set_title(f'Avg PSFcam at best bin {best_bin}  (N={nframes[best_bin]})')
ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Show the averaged PLcam image at the best-covered bin
fig, ax = plt.subplots(figsize=(10, 4))
img = avg_plcam[best_bin[0], best_bin[1]]
ax.imshow(img, aspect='auto', origin='lower', cmap='gray',
          vmin=np.nanpercentile(img, 5), vmax=np.nanpercentile(img, 99))
ax.set_title(f'Averaged PLcam at best bin {best_bin}  (N={nframes[best_bin]} frames)')
ax.set_xlabel('Spectral axis (pixels)')
ax.set_ylabel('Fiber axis (pixels)')
plt.tight_layout()
plt.show()
print('Each horizontal stripe is one fiber. The spectral dimension runs left-right.')

## Summary

Step 3 produced `step3_map.h5` with shape `(map_n, map_n, ny, nx)` — one averaged PLcam image per sky position.

The coupling map shows how much light each fiber collects as a function of where the PSF is pointing. This is the core measurement we need for image reconstruction.

**Next**: Run `tutorial_step4_specextract.ipynb` to extract per-fiber spectra from these averaged frames, producing a FITS coupling map.

---

## CLI equivalent

```ini
[Average]
input       = tutorial_output_new/step2_alldata.h5
map_n       = 5
map_width   = 2.0
xc          = 18.26
yc          = 20.50
pix2mas     = 16.2
n_bootstrap = 0
output      = tutorial_output_new/step3_map.h5
```

```python
import PLred.average as average
average.average_to_h5_from_config('obs.ini')
# or:
import PLred.pipeline as pipeline
pipeline.run_mode1('obs.ini', steps=[3])
```

In [ ]:
print('Done. Step 3 output saved to:', OUTPUT_H5)